# 分散 Self-Play — Kaggle worker

試合を生成してシャードを1ファイル出すだけ。学習はしない。**GPU 不要・インターネット不要。**

入力: Dataset `ptcg-repo`(`ptcg_repo.zip`)と `ptcg-run-<run 名>`(`run_*.zip`)
出力: `/kaggle/working/<worker 名>.npz` — Notebook の Output からダウンロードする

Save & Run All にすれば、ブラウザを閉じても裏で最後まで走る。


## 1. 環境確認と展開


In [ ]:
!python -V
!nproc


In [ ]:
import glob, json, os, shutil, zipfile

EXPECTED_GEN = -1   # push_kaggle.py が実際の世代に書き換える
EXPECTED_RUN_ID = ''   # push_kaggle.py が実際の run 名に書き換える

# まず入力に何が来ているかを必ず出す。Kaggle は Dataset にアップロードした zip を
# 展開して置くことがあるので、zip のままでも展開済みでも動くようにする。
print('--- /kaggle/input ---')
for p in sorted(glob.glob('/kaggle/input/**', recursive=True))[:60]:
    print(' ', p)

os.makedirs('/kaggle/temp/ptcg', exist_ok=True)

def _place(zip_glob, marker_glob, marker_depth, dest):
    """zip があれば展開、無ければ展開済みの場所を marker から探してコピー。"""
    z = glob.glob(zip_glob, recursive=True)
    if z:
        print('zip から展開:', z[0])
        zipfile.ZipFile(z[0]).extractall(dest)
        return
    m = glob.glob(marker_glob, recursive=True)
    if not m:
        raise SystemExit(f'入力が見つからない: {zip_glob} も {marker_glob} も無い。'
                         'Dataset が Notebook に添付されているか確認する。')
    root = m[0]
    for _ in range(marker_depth):
        root = os.path.dirname(root)
    print('展開済みをコピー:', root)
    shutil.copytree(root, dest, dirs_exist_ok=True)

_place('/kaggle/input/**/ptcg_repo.zip',
       '/kaggle/input/**/sample_submission/cg/libcg.so', 3, '/kaggle/temp/ptcg')
# run zip は中に run/ を含むので展開先は /kaggle/temp。展開済みの場合は run/ 自体をコピー。
if glob.glob('/kaggle/input/**/run_*.zip', recursive=True):
    _place('/kaggle/input/**/run_*.zip', '', 0, '/kaggle/temp')
else:
    _place('/nonexistent', '/kaggle/input/**/run/run.json', 1, '/kaggle/temp/run')


# 必要なものが本当に来ているかを名指しで確認する(Dataset の版が古いと欠ける)
need = ['kaggle_replays/rl/distributed/worker.py',
        'kaggle_replays/rl/collect_parallel.py',
        'sample_submission/cg/libcg.so',
        'kaggle_replays/meta_analysis/archetype_decks/dragapult_ex/01.csv']
missing = [n for n in need if not os.path.exists('/kaggle/temp/ptcg/' + n)]
print('\n--- 必要ファイルの確認 ---')
for n in need:
    print(('  OK  ' if n not in missing else '  なし '), n)
if missing:
    raise SystemExit('Dataset の中身が足りない(古い版が添付された可能性)。'
                     f'欠けているもの: {missing}')


# Dataset の新バージョンが反映される前に実行されると、1つ前の世代の run.json が
# 添付される。ファイルは揃っているので存在チェックでは気づけない。ここで世代を
# 突き合わせて即座に落とす(3分かけて無駄な経験を集めてしまうのを防ぐ)。
run_cfg = json.load(open('/kaggle/temp/run/run.json', encoding='utf-8'))
print('run.json:', run_cfg['run_id'], 'v%d' % run_cfg['generation'],
      '/ 期待:', EXPECTED_RUN_ID, 'v%d' % EXPECTED_GEN)
if EXPECTED_GEN >= 0 and run_cfg['generation'] != EXPECTED_GEN:
    raise SystemExit(f"Dataset が古い: run.json は v{run_cfg['generation']} だが "
                     f"v{EXPECTED_GEN} を期待。新バージョンがまだ反映されていない。")

# run を複数並行で回すときの取り違え。Dataset は run ごとに分けてあるが、
# 添付先を間違えると別の run のモデルで学習してしまう。世代番号だけでは見抜けない。
if EXPECTED_RUN_ID and run_cfg['run_id'] != EXPECTED_RUN_ID:
    raise SystemExit(f"別の run の Dataset が添付されている: run.json は "
                     f"{run_cfg['run_id']} だが {EXPECTED_RUN_ID} を期待。")

!ls /kaggle/temp/run /kaggle/temp/run/models


## 2. ゲームエンジンが動くか


In [ ]:
%cd /kaggle/temp/ptcg/kaggle_replays/rl
!python test_rollout.py


## 3. 並列処理の起動方式を確定させる


In [ ]:
# Linux での並列処理の起動方式を確かめる。
#
# 手元(Windows)は spawn が既定なので検証済みだが、Linux の既定は fork で、
# 親が読み込み済みの cg エンジン(ネイティブライブラリ)を子が引き継ぐ。ここで
# 両方を実際に走らせて、どちらが使えるかを確定させる。
import subprocess, sys, os

os.chdir('/kaggle/temp/ptcg/kaggle_replays/rl/distributed')

# 診断用に試合数の少ない run を作る(本番の run とは別物)
subprocess.run([sys.executable, 'init_run.py', '--run-id', 'diag',
                '--run-dir', '/kaggle/temp/diagrun', '--workers', 'w0',
                '--games-per-worker', '8'], check=True)

for method in ('spawn', 'fork'):
    print(f'\n===== start-method = {method} =====', flush=True)
    r = subprocess.run(
        [sys.executable, 'worker.py', '--run-dir', '/kaggle/temp/diagrun',
         '--worker-id', 'w0', '--workers', '2', '--start-method', method,
         '--out', f'/tmp/diag_{method}.npz'],
        capture_output=True, text=True, timeout=1800)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('--- stderr ---'); print(r.stderr[-2000:])
    print(f'>>> {method}: ' + ('OK' if r.returncode == 0 else f'NG (exit {r.returncode})'))


## 4. 本番の収集

シャードは `/kaggle/working/` へ直接書き出す(Notebook の Output になる)。


In [ ]:
%cd /kaggle/temp/ptcg/kaggle_replays/rl/distributed
!python worker.py --run-dir /kaggle/temp/run --worker-id kaggle --workers 4 \
    --start-method spawn --out /kaggle/working/kaggle.npz
!ls -lh /kaggle/working/
